# __Demo: Structured Prompts with the Four-Element Framework for Engineering Tasks__

## __Steps to Perform:__
Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`

Step 2: Define a Function to Get Completion

Step 3: Write 10 Structured Prompts Using the Four-Element Framework

Step 4: Apply Zero-Shot Prompting to Two Engineering Tasks

Step 5: Apply Few-Shot Prompting to a Third Engineering Task

### __Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`__
- Installs required libraries (`transformers`, `torch`, `accelerate`).
- Initializes `Qwen/Qwen2.5-0.5B-Instruct` directly onto the GPU in 16-bit half precision (`torch.float16`).
- Requires no Hugging Face API key, token, or billing configuration.

In [1]:
!pip install -q transformers torch accelerate

import torch
from transformers import pipeline

# Initialize text-generation pipeline on GPU with float16
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.


### __Step 2: Define a Function to Get Completion__
The __get_completion__ function sends a prompt to the model formatted as a chat message and returns its response using greedy decoding (`do_sample=False`) for deterministic software engineering outputs.

In [2]:
def get_completion(prompt, max_tokens=150):
    messages = [{"role": "user", "content": prompt}]
    output = generator(
        messages,
        max_new_tokens=max_tokens,
        do_sample=False
    )
    return output[0]["generated_text"][-1]["content"].strip()

print("✅ Completion helper ready.")

✅ Completion helper ready.


### __Step 3: Write 10 Structured Prompts Using the Four-Element Framework__
Every prompt below is built from the same four elements taught in this lesson:

- __Instruction__: the specific task for the model
- __Context__: the surrounding situation that shapes the answer
- __Input data__: the concrete code, error, or text the model must work with
- __Output indicator__: the exact format the response must take

Each entry below targets a real engineering task rather than a generic example.

In [3]:
structured_prompts = [
    {
        "task": "Scaffold an endpoint",
        "instruction": "Write a Python function.",
        "context": "It will be used as a route handler in a Flask web service.",
        "input_data": "The function must return the current server time in ISO 8601 format.",
        "output_indicator": "Return only the function code, with no explanation.",
    },
    {
        "task": "Fix a naming bug",
        "instruction": "Refactor this function.",
        "context": "The function name shadows a Python built-in, which can cause subtle bugs elsewhere in the codebase.",
        "input_data": "def list():\n    return [1, 2, 3]",
        "output_indicator": "Return only the corrected function.",
    },
    {
        "task": "Spot an edge case",
        "instruction": "Identify the bug in this function.",
        "context": "The function is meant to return the average of a list of numbers.",
        "input_data": "def avg(nums):\n    return sum(nums) / len(nums)",
        "output_indicator": "Answer in exactly one sentence naming the edge case that breaks it.",
    },
    {
        "task": "Write a unit test",
        "instruction": "Write a pytest unit test for this function.",
        "context": "The function checks whether an integer is even.",
        "input_data": "def is_even(n):\n    return n % 2 == 0",
        "output_indicator": "Return a single pytest test function, nothing else.",
    },
    {
        "task": "Write a docstring",
        "instruction": "Write a Google-style docstring for this function.",
        "context": "The function validates an email address using a regular expression.",
        "input_data": "def is_valid_email(s):\n    return re.match(r'^[^@]+@[^@]+\\.[^@]+$', s) is not None",
        "output_indicator": "Return only the docstring text, including Args and Returns sections.",
    },
    {
        "task": "Summarize an error",
        "instruction": "Summarize this error for a non-technical stakeholder.",
        "context": "This error appeared in the application logs for the alerts API.",
        "input_data": "KeyError: 'user_id'",
        "output_indicator": "One plain-language sentence, no jargon.",
    },
    {
        "task": "Fix a security issue",
        "instruction": "Suggest a corrected version of this line of code.",
        "context": "This SQL query is built with string formatting and is vulnerable to SQL injection.",
        "input_data": 'query = f"SELECT * FROM users WHERE name = \'{name}\'"',
        "output_indicator": "Return only the corrected line of code, using parameterized queries.",
    },
    {
        "task": "Write a commit message",
        "instruction": "Write a git commit message for this change.",
        "context": "The change fixes a KeyError that occurred when a lookup dictionary was missing a key.",
        "input_data": "No input data required.",
        "output_indicator": "One line, imperative mood, under 72 characters.",
    },
    {
        "task": "Explain code to a new engineer",
        "instruction": "Explain what this code does.",
        "context": "The reader is a new engineer who has not used Python decorators before.",
        "input_data": "@lru_cache(maxsize=None)\ndef fib(n):\n    return n if n < 2 else fib(n - 1) + fib(n - 2)",
        "output_indicator": "Exactly two sentences, in plain language.",
    },
    {
        "task": "Write a deployment checklist item",
        "instruction": "Write one deployment checklist item.",
        "context": "The team is rolling out a change to an API's rate-limiting configuration.",
        "input_data": "No input data required.",
        "output_indicator": "One bullet point, starting with a verb.",
    },
]

def build_structured_prompt(entry):
    return (
        f"Instruction: {entry['instruction']}\n"
        f"Context: {entry['context']}\n"
        f"Input data: {entry['input_data']}\n"
        f"Output indicator: {entry['output_indicator']}"
    )

for entry in structured_prompts:
    prompt = build_structured_prompt(entry)
    print(f"=== {entry['task']} ===")
    print(prompt)
    print("--- Response ---")
    print(get_completion(prompt, max_tokens=100))
    print()


=== Scaffold an endpoint ===
Instruction: Write a Python function.
Context: It will be used as a route handler in a Flask web service.
Input data: The function must return the current server time in ISO 8601 format.
Output indicator: Return only the function code, with no explanation.
--- Response ---
```python
import datetime

def get_current_server_time():
    """
    This function returns the current server time in ISO 8601 format.

    Returns:
        str: The current server time in ISO 8601 format.
    """
    now = datetime.datetime.now()
    iso_time = now.isoformat()
    return iso_time
```

This Python function `get_current_server_time` uses the built-in `datetime` module to retrieve and return the current server time in

=== Fix a naming bug ===
Instruction: Refactor this function.
Context: The function name shadows a Python built-in, which can cause subtle bugs elsewhere in the codebase.
Input data: def list():
    return [1, 2, 3]
Output indicator: Return only the correcte

### __Step 4: Apply Zero-Shot Prompting to Two Engineering Tasks__
Zero-shot means the model is given no examples -- only a clear instruction. Below, this is applied to two engineering tasks: triaging a bug report's severity, and writing a commit message from a change summary.

In [4]:
# Engineering task 1: zero-shot bug severity triage
triage_prompt = """
Classify the severity of this bug report as Critical, High, Medium, or Low. Respond with only the severity label and a one-sentence justification.

Bug report: "The /payments/charge endpoint intermittently returns a 500 error under load. Roughly 1 in 200 requests fail. Retrying the same request usually succeeds. No data loss has been observed."
"""
print("=== Engineering Task 1: Bug Severity Triage ===")
print(get_completion(triage_prompt, max_tokens=60))


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


=== Engineering Task 1: Bug Severity Triage ===
Critical


In [5]:
# Engineering task 2: zero-shot commit message generation
commit_prompt = """
Write a git commit message for the following change. Use the imperative mood and keep the first line under 72 characters.

Change summary: Added a null check before accessing alert.resolved_by in the explain endpoint, which was raising an AttributeError whenever an alert had not yet been acknowledged.
"""
print("\n=== Engineering Task 2: Commit Message Generation ===")
print(get_completion(commit_prompt, max_tokens=50))



=== Engineering Task 2: Commit Message Generation ===
"Added null check to ensure 'resolved_by' is accessed correctly when handling alerts."


### __Step 5: Apply Few-Shot Prompting to a Third Engineering Task__
Few-shot means the model is given a small number of labeled examples before the new case. Below, this is applied to classifying a pull request review comment as either __Blocking__ or __Nit__.

In [6]:
few_shot_prompt = """
Classify each PR review comment as either Blocking or Nit.

Comment: "This will throw a NullReferenceException if the user has no email on file."
Classification: Blocking

Comment: "Minor: this variable could be named more descriptively."
Classification: Nit

Comment: "This endpoint has no authentication check -- anyone can call it."
Classification: Blocking

Comment: "Consider adding a blank line here for readability."
Classification: Nit

Comment: "This query builds SQL with string formatting instead of parameters."
Classification:
"""
print("=== Engineering Task 3: Few-Shot PR Comment Classification ===")
print(get_completion(few_shot_prompt, max_tokens=30))


=== Engineering Task 3: Few-Shot PR Comment Classification ===
Classification: Blocking


## __Conclusion__
This demo builds 10 structured prompts from the four-element framework, each aimed at a real engineering task rather than a generic example, and then applies zero-shot prompting to two further tasks and few-shot prompting to a third. Together, this satisfies the lesson's workshop: writing structured prompts with the four-element framework and applying zero-shot and few-shot techniques to engineering tasks.